In [ ]:
import numpy as np
from tqdm import tqdm
import json

from lac.perception.segmentation import SemanticClasses, UnetSegmentation
from lac.perception.segmentation_util import color_to_label, label_to_color
from lac.util import load_data, load_images
from lac.utils.visualization import image_grid
from lac.params import LAC_DATA_PATH

%load_ext autoreload
%autoreload 2

## Load data


In [ ]:
# run_name = "semantics_map1_preset2_recovery_agent"
run_name = "semantics_map1_preset7_beached"
data_path = LAC_DATA_PATH / "segmentation" / run_name
initial_pose, lander_pose, poses, imu_data, cam_config, json_data = load_data(data_path)
config = json.load(open("../../configs/nine_loops.json"))
print(f"Loaded {len(poses)} poses")

In [ ]:
START_FRAME = 100
END_FRAME = 4000
images = load_images(
    data_path,
    cameras=["FrontLeft", "FrontRight", "FrontLeft_semantic", "FrontRight_semantic"],
    start_frame=START_FRAME,
    end_frame=END_FRAME,
)

## Run segmentation


In [ ]:
segmentation = UnetSegmentation()

In [ ]:
FRAME = 100

img = images["FrontLeft"][FRAME]
pred_label = segmentation.predict(img)
pred_color = label_to_color(pred_label)
gt_color = images["FrontLeft_semantic"][FRAME]
gt_label = color_to_label(gt_color)

In [ ]:
pred_vis = label_to_color(pred_label, custom=True)
gt_vis = label_to_color(gt_label, custom=True)

In [ ]:
error = pred_label != gt_label
error_vis = 255 * np.ones((*error.shape, 3), dtype=np.uint8)
error_vis[error] = (255, 0, 0)  # Red where there are errors

In [ ]:
image_grid([img, gt_vis, pred_vis, error_vis], rows=1, cols=4, figsize=(60, 10))

### Analysis


In [ ]:
rock_true_positives = 0
rock_false_positives = 0
rock_false_negatives = 0
rock_true_negatives = 0

for frame in tqdm(range(START_FRAME, 1000, 2)):
    img = images["FrontLeft"][frame]
    pred_label = segmentation.predict(img)
    gt_color = images["FrontLeft_semantic"][frame]
    gt_label = color_to_label(gt_color)

    # Create binary masks for rock class
    pred_rock_mask = pred_label == SemanticClasses.ROCK.value
    gt_rock_mask = gt_label == SemanticClasses.ROCK.value

    # Compute confusion matrix components
    rock_true_positives += np.sum(pred_rock_mask & gt_rock_mask)
    rock_false_positives += np.sum(pred_rock_mask & ~gt_rock_mask)
    rock_false_negatives += np.sum(~pred_rock_mask & gt_rock_mask)
    rock_true_negatives += np.sum(~pred_rock_mask & ~gt_rock_mask)

# Calculate rates
rock_false_positive_rate = rock_false_positives / (rock_false_positives + rock_true_negatives)
rock_false_negative_rate = rock_false_negatives / (rock_false_negatives + rock_true_positives)

print(f"Rock False Positive Rate: {rock_false_positive_rate:.4f}")
print(f"Rock False Negative Rate: {rock_false_negative_rate:.4f}")
print(
    f"Total pixels analyzed: {rock_true_positives + rock_false_positives + rock_false_negatives + rock_true_negatives}"
)